# Torch vs ONNX performance

In [6]:
import torch
import onnx
import onnxruntime
import lenet
import numpy as np

torch.manual_seed(42)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Test dataset

In [7]:
X = lenet.load_tensor(lenet.sample_filename)
y = lenet.load_tensor(lenet.labels_filename, dtype="<i8")

X.shape, y.shape

Tensor shape = (64, 1, 28, 28)
Tensor shape = (64,)


((64, 1, 28, 28), (64,))

## PyTorch model

In [8]:
lenet_model = lenet.LeNetModel().to(device)
lenet_model.load_state_dict(torch.load(lenet.torch_filename, map_location=device))
lenet_model.eval()


LeNetModel(
  (lenet_stack): Sequential(
    (0): Conv2d(1, 12, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1152, out_features=120, bias=True)
    (8): ReLU()
    (9): Linear(in_features=120, out_features=84, bias=True)
    (10): Sigmoid()
    (11): Linear(in_features=84, out_features=10, bias=True)
  )
)

## ONNX

In [9]:
onnx_model = onnx.load(lenet.onnx_filename)
onnx.checker.check_model(onnx_model)
print(f"onnx op_set version={onnx.defs.onnx_opset_version()}")
print(f"onnx_model.op_set_version={onnx_model.opset_import[0].version}")
print(f"onnx_model.ir_version={onnx_model.ir_version}")

model_input = onnx_model.graph.input[0]
print(f"onnx_model.input: name={model_input.name}, type={model_input.type.tensor_type.elem_type} shape={model_input.type.tensor_type.shape.dim}")

model_output = onnx_model.graph.output[0]
print(f"onnx_model.output: name={model_output.name}, type={model_output.type.tensor_type.elem_type} shape={model_output.type.tensor_type.shape.dim}")


onnx op_set version=28
onnx_model.op_set_version=20
onnx_model.ir_version=10
onnx_model.input: name=images, type=1 shape=[dim_value: 1
, dim_value: 1
, dim_value: 28
, dim_value: 28
]
onnx_model.output: name=logits, type=1 shape=[dim_value: 1
, dim_value: 10
]


## ONNX C++ inference

In [10]:
import subprocess
from pathlib import Path

project_root = Path(lenet.__file__).resolve().parent.parent
subprocess.run(
    [str(project_root / "build/onnx_exporter"), "model/lenet.onnx"],
    cwd=project_root,
    check=True,
)


ONNX Runtime v1.30.1
Available providers:
  CPUExecutionProvider
Running inference
[-3.0544376-3.0544376,-1.3292674-1.3292674,-1.513484-1.513484,-0.630889-0.630889,-7.1123204-7.1123204,12.54264112.542641,-1.1275439-1.1275439,6.06174956.0617495,-1.3873167-1.3873167,1.36275461.3627546]
[-2.0879903-2.0879903,-3.2771573-3.2771573,-0.85679275-0.85679275,2.34380582.3438058,-7.2599554-7.2599554,3.43274953.4327495,-5.868806-5.868806,12.50523212.505232,2.47557072.4755707,1.38807451.3880745]
[-0.051923588-0.051923588,-0.24464889-0.24464889,4.8108744.810874,3.69565683.6956568,10.38751410.387514,-8.942588-8.942588,3.98206853.9820685,-4.844415-4.844415,-2.529093-2.529093,-4.9680457-4.9680457]
[-3.588472-3.588472,-1.2102939-1.2102939,-1.5069826-1.5069826,0.777282660.77728266,-8.781161-8.781161,5.7475295.747529,-5.7316813-5.7316813,12.60732212.607322,1.10114541.1011454,3.96128343.9612834]
[-0.25817028-0.25817028,-2.2764308-2.2764308,4.197584.19758,6.3758996.375899,6.89933066.8993306,-8.720177-8.72017

CompletedProcess(args=['/home/arykovanov/src/learning/arm_preparation/build/onnx_exporter', 'model/lenet.onnx'], returncode=0)

In [11]:
cpp_logits = lenet.load_tensor(lenet.cpp_logits_filename)
cpp_logits.shape

Tensor shape = (64, 10)


(64, 10)

## Logit values

Run inference with both PyTorch and ONNX Runtime for all the presaved image batch

In [12]:
onnxruntime_session = onnxruntime.InferenceSession(lenet.onnx_filename)

torch_logits = torch.zeros((len(X), 10))
onnx_logits = torch.zeros((len(X), 10))

with torch.no_grad():
    for idx in range(len(y)):
        img = X[idx]
        label = y[idx]
        pred_onnx = onnxruntime_session.run(None, {"images": img.reshape(1, *img.shape)})[0].reshape(-1)
        pred_torch = lenet_model(torch.from_numpy(img.reshape(1, 1, 28, 28)))
        torch_logits[idx] = pred_torch
        onnx_logits[idx] = torch.from_numpy(pred_onnx)


Check if ONNX and Pytorch prediction are the same.

In [13]:
print(f"label: {y[0]}")
print(f"torch: {torch_logits[0]}")
print(f"onnx: {onnx_logits[0]}")
print(f"cpp: {cpp_logits[0]}")

for idx in range(len(y)):
    label = y[idx]
    # print(np.abs(onnx_logits[idx] - torch_logits[idx]).max())
    if torch_logits[idx].argmax() != onnx_logits[idx].argmax():
        print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={torch_logits[idx].argmax()}, pred_onnx={onnx_logits[idx].argmax()}")
    if torch_logits[idx].argmax() != cpp_logits[idx].argmax():
        print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={torch_logits[idx].argmax()}, pred_cpp={cpp_logits[idx].argmax()}")
    if abs(onnx_logits[idx] - torch_logits[idx]).max() > 1e-5:
        raise ValueError(f"Logits at sample {idx}: label={label}, pred_torch={torch_logits[idx]}, pred_onnx={onnx_logits[idx]}. The difference is {np.abs(onnx_logits[idx] - torch_logits[idx]).max()} and it is TOO BIG! The difference should be less than 1e-5.")


label: 5
torch: tensor([-3.0544, -1.3293, -1.5135, -0.6309, -7.1123, 12.5426, -1.1275,  6.0618,
        -1.3873,  1.3628])
onnx: tensor([-3.0544, -1.3293, -1.5135, -0.6309, -7.1123, 12.5426, -1.1275,  6.0617,
        -1.3873,  1.3628])
cpp: [-3.0544376 -1.3292674 -1.513484  -0.630889  -7.1123204 12.542641
 -1.1275439  6.0617495 -1.3873167  1.3627546]
